## BM25 on MINI CORPUS

In [4]:
# Example mini corpus
corpus = [
    "Machine learning for cancer diagnosis",
    "Deep learning MRI diagnosis",
    "Cancer treatment and prevention",
    "Machine learning models for medical imaging",
    "Diagnosis of skin cancer using machine learning"
]

# Query
query = "machine learning cancer diagnosis"

In [19]:
%pip install rank_bm25 sentence_transformers

In [21]:
from rank_bm25 import BM25Okapi   # BM25Okapi is the specific search formula we want to use

In [6]:
# Simple whitespace tokenizer
'''
take every sentence in the corpus and the query,
convert all the letters to lowercase (.lower()), and break them into
individual words based on the spaces between them (.split())
'''
tokenized_corpus = [doc.lower().split() for doc in corpus]
tokenized_query = query.lower().split()

In [7]:
#Initialize BM25 and get scores
'''
Here we look at the five docs and give each doc a score based on how well it matches the query terms.
'''
bm25 = BM25Okapi(tokenized_corpus)   # feed the broken individual words into BM25
scores = bm25.get_scores(tokenized_query)  # grade each doc (bm25) based on the broken up query

In [8]:
for i, s in enumerate(scores):     # loop through the scores and print them upto three decimal places
    print(f"Doc {i+1} → Score: {s:.3f} | {corpus[i]}")

Doc 1 → Score: 0.656 | Machine learning for cancer diagnosis
Doc 2 → Score: 0.360 | Deep learning MRI diagnosis
Doc 3 → Score: 0.180 | Cancer treatment and prevention
Doc 4 → Score: 0.301 | Machine learning models for medical imaging
Doc 5 → Score: 0.558 | Diagnosis of skin cancer using machine learning


In [9]:
import numpy as np

ranking = np.argsort(scores)[::-1]   # find the positions of the highest scores and organize them from best to worst.
print("\nRanked Documents:")
for rank, idx in enumerate(ranking, start=1):
    print(f"{rank}. ({scores[idx]:.3f}) {corpus[idx]}")


Ranked Documents:
1. (0.656) Machine learning for cancer diagnosis
2. (0.558) Diagnosis of skin cancer using machine learning
3. (0.360) Deep learning MRI diagnosis
4. (0.301) Machine learning models for medical imaging
5. (0.180) Cancer treatment and prevention


## BM25 ON 3  PDFs

In [1]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.0/349.0 kB 6.7 MB/s eta 0:00:00


In [10]:
import numpy as np
from pypdf import PdfReader
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, util

In [11]:
pdf_files = ["learning-langchain-for-true-epub-9781098167288.pdf", "Natural-Language-Processing-Python.pdf", "test.pdf"]

query = "What are the different types of tokenizers"

In [12]:
# HELPER FUNCTION: READ AND CHUNK PDFS
def extract_and_chunk_pdf(file_path):
    reader = PdfReader(file_path)
    full_text = ""
    chunks = []

    for page in reader.pages:
        text = page.extract_text()
        if text:
            full_text += text + "\n"

            # Split the page into smaller "parts"
            # split by double newline and ignore very short random lines
            page_chunks = [p.strip() for p in text.split('\n\n') if len(p.strip()) > 20]
            chunks.extend(page_chunks)

    return full_text, chunks

In [13]:
# Read all PDFs and store their data
documents = []
for file in pdf_files:
    full_text, chunks = extract_and_chunk_pdf(file)
    documents.append({
        "filename": file,
        "full_text": full_text,
        "chunks": chunks
    })

In [14]:
# Tokenize the query and the full text of all 3 PDFs
tokenized_query = query.lower().split()
tokenized_full_docs = [doc["full_text"].lower().split() for doc in documents]

In [15]:
# Run BM25 on the full documents
bm25_docs = BM25Okapi(tokenized_full_docs)
doc_scores = bm25_docs.get_scores(tokenized_query)

In [16]:
# FIND THE BEST MATCHING PDF

best_doc_idx = np.argmax(doc_scores)
winning_pdf = documents[best_doc_idx]

print(f"Best matching PDF: {winning_pdf['filename']} (Score: {doc_scores[best_doc_idx]:.3f})")

Best matching PDF: test.pdf (Score: 1.007)


In [17]:
# FIND THE EXACT PART IN THE PDF

# Tokenize the individual PARTS of the winning PDF
tokenized_chunks = [chunk.lower().split() for chunk in winning_pdf["chunks"]]

# Run BM25 again, but only on the parts of the best pdf
bm25_chunks = BM25Okapi(tokenized_chunks)
chunk_scores = bm25_chunks.get_scores(tokenized_query)

# Find the part
best_chunk_idx = np.argmax(chunk_scores)
winning_chunk = winning_pdf["chunks"][best_chunk_idx]

print(f"Best Part (Score: {chunk_scores[best_chunk_idx]:.3f}):",winning_chunk)

Best Part (Score: 14.646): A tour of real-world pretrained tokenizers (from BERT to GPT-2, GPT-4,
and other models) showed us areas where some tokenizers are better (e.g.,
preserving information like capitalization, newlines, or tokens in other
languages) and other areas where tokenizers are just different from each
other (e.g., how they break down certain words).
Three of the major tokenizer design decisions are the tokenizer algorithm
(e.g., BPE, WordPiece, SentencePiece), tokenization parameters (including
vocabulary size, special tokens, capitalization, treatment of capitalization
and different languages), and the dataset the tokenizer is trained on.
Language models are also creators of high-quality contextualized token
embeddings that improve on raw static embeddings. Those contextualized
token embeddings are what’s used for tasks including named-entity
recognition (NER), extractive text summarization, and text classification. In
addition to producing token embeddings, language mo

In [23]:
# MiniLM Semantic Retrieval
model = SentenceTransformer("all-MiniLM-L6-v2")

# Extract the full text content and filenames from the documents list
document_texts = [doc["full_text"] for doc in documents]
document_names = [doc["filename"] for doc in documents]

doc_embeddings = model.encode(document_texts, convert_to_tensor=True)

# Assuming 'query' from previous cells is the intended input query
queries = [query]

for q in queries:
    q_emb = model.encode(q, convert_to_tensor=True)
    sims = util.cos_sim(q_emb, doc_embeddings)[0]
    ranking = sorted(zip(document_names, sims.tolist()), key=lambda x: x[1], reverse=True)
    print(f"Query: {q}")
    for name, s in ranking:
        print(f"  {s:.4f}  {name}")
    print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Query: What are the different types of tokenizers
  0.2442  test.pdf
  0.2279  learning-langchain-for-true-epub-9781098167288.pdf
  0.2078  Natural-Language-Processing-Python.pdf



In [25]:
import re

# Split every PDF into sentences, remembering which file each came from.
passages, passage_src = [], []
for name, doc in zip(document_names, documents):
    # Extract the full text from the document dictionary for sentence splitting
    full_text_doc = doc["full_text"]
    for sent in re.split(r"(?<=[.!?])\s+", full_text_doc):
        sent = sent.strip()
        if len(sent.split()) >= 4:        # skip tiny fragments
            passages.append(sent)
            passage_src.append(name)

passage_emb = model.encode(passages, convert_to_tensor=True)

for q in queries:
    sims = util.cos_sim(model.encode(q, convert_to_tensor=True), passage_emb)[0]
    top = sims.topk(2)
    print(f"Query: {q}")
    for score, i in zip(top.values, top.indices):
        i = int(i)
        print(f"  [{score:.3f}] ({passage_src[i]}) {passages[i]}")
    print()

Query: What are the different types of tokenizers
  [0.774] (test.pdf) Recall from Chapter 2 that the tokenizer contains a table of tokens—the
tokenizer’s vocabulary.
  [0.767] (test.pdf) Three of the major tokenizer design decisions are the tokenizer algorithm
(e.g., BPE, WordPiece, SentencePiece), tokenization parameters (including
vocabulary size, special tokens, capitalization, treatment of capitalization
and different languages), and the dataset the tokenizer is trained on.

